## Scaling the LIF

`02_graph.ipynb` showed *a* path exists from `ol_sensory` to `vnc_motor`, and roughly how strong each link is, but a graph only shows structure, it says nothing about whether a real signal would actually survive the trip. that's a dynamics question, not a structural one.

this file takes the real, found path from `02_graph.ipynb` and runs it through the same leaky-integrate-and-fire mechanics from `01_lif.ipynb`, but with real weights this time instead of an arbitrary toy cluster, to see whether a pulse injected at the eye actually reaches the motor neuron, and what that reveals about timing.

### Init

In [23]:
import pandas as pd
from pathlib import Path

root = Path.cwd().parent
# root = Path(__file__).resolve().parent.parent

neurons_file = root / 'data' / 'body-annotations-male-cns-v1.0-minconf-0.5.feather'
filtered_path = root / 'data' / 'pipeline_edges_filtered.feather'

df = pd.read_feather(neurons_file)
id_to_superclass = df[df['status'] == 'Traced'].set_index('bodyId')['superclass'].to_dict()

strong_only = pd.read_feather(filtered_path)

#### Loading the path and its edges

the path found in `02_graph.ipynb`, hardcoded here since the search is deterministic, same source and target would always produce the same result

In [24]:
path = [14428, 13984, 10286, 11974, 10680, 164190]

pull every edge where BOTH ends are one of these 6 neurons, not just the 5 consecutive hops along the path, since these neurons may also connect to each other in ways the shortest-path search didn't use (shortcuts, feedback)

In [25]:
path_edges = strong_only[
    strong_only['body_pre'].isin(path) & strong_only['body_post'].isin(path)
]
print(path_edges)

          body_pre  body_post  weight             pre_sc            post_sc
6322824      10680      11974       5  descending_neuron       cb_intrinsic
6324876      10680     164190      78  descending_neuron          vnc_motor
10227568     14428      13984      74         ol_sensory  visual_projection
11913142     10286      13984       9  descending_neuron  visual_projection
11913888     10286      11974      92  descending_neuron       cb_intrinsic
11930105     10286     164190      10  descending_neuron          vnc_motor
13407577     11974      10286       4       cb_intrinsic  descending_neuron
13408386     11974      10680      41       cb_intrinsic  descending_neuron
17544937     13984      10286      45  visual_projection  descending_neuron


> reminder: the theorized path was along the lines of
>
> `ol_sensory -> visual_projection -> cb_intrinsic -> descending_neuron -> vnc_motor`
>
> there's a shortcut, `10286 -> 164190` (weight 10), skipping two hops, and there are weak feedback edges running backward against the theorized `ol -> vnc` pipeline order: `10680 -> 11974` (weight 5) and `10286 -> 13984` (weight 9)
>
> the path itself already contains the real surprise from `02_graph.ipynb`: `10286 -> 11974` (weight 92) goes backward from `descending_neuron` to `cb_intrinsic`, against the theorized direction, before continuing on to `10680` and finally `164190`. that's the loop, not an extra edge sitting outside the path
>
> the weakest edge along the actual path is `11974 -> 10680` at weight 41, this is the bottleneck that decides what threshold is even possible (more on this later)

#### Setting up the simulation

same structures as `01_lif.ipynb`, just built from the real path and its edges instead of a hand-picked cluster

In [26]:
neuron_ids = path
charge = {nid: 0 for nid in neuron_ids}
edges = list(path_edges[['body_pre', 'body_post', 'weight']].itertuples(index=False, name=None))

#### Parameters

none of these were picked correctly on the first try, each one got adjusted at least once after seeing what the simulation actually did:

- **`threshold`**: how much charge a neuron needs before it fires. too low and every pulse trivially succeeds, nothing to learn. the useful value sits just above the path's weakest edge, so a single pulse alone fails, forcing the real question of whether two pulses together can succeed
- **`decay_factor`**: how much charge leaks away each tick. this controls how long a neuron "remembers" a pulse that didn't quite reach threshold, worth comparing a couple of different values to see how much the leftover, and the resulting timing window, actually changes
- **`time_steps`**: how many ticks the simulation runs for. has to cover the full travel time of every pulse injected, including a second, later pulse, a run that's cut short can look like a failure when the signal just hadn't arrived yet
- **`external_input`**: when and where a pulse gets injected. testing one isolated pulse versus two pulses at different gaps apart is what actually reveals the timing window, not a single configuration

the iterations below show how these were tuned, what failed, and why, before landing on the values that actually demonstrate the window

#### Running the simulation

Iteration 1. threshold = 40, single pulse, baseline check

In [ ]:
# reset for future iterations
charge = {nid: 0 for nid in neuron_ids}

decay_factor = 0.8
threshold = 40
time_steps = 8
external_input = {14428: 100}

for t in range(time_steps):
    print(f"--- time step {t+1} ---")

    # apply external input at tick 0
    if t == 0:
        for nid, amount in external_input.items():
            print("external applied to", nid)
            charge[nid] += amount

    # figure out who fires this tick, based on charge BEFORE any resets happen
    fired = [nid for nid in neuron_ids if charge[nid] >= threshold]
    print("fired:", fired)

    # apply the leak to everyone
    for nid in neuron_ids:
        charge[nid] *= decay_factor

    # firing neurons dump their charge to neighbors, then reset to 0
    for pre, post, weight in edges:
        if pre in fired:
            print(f"{pre} fires away to {post}")
            charge[post] += weight

    for nid in fired:
        charge[nid] = 0

    print({nid: round(c, 2) for nid, c in charge.items()})

--- time step 1 ---
external applied to 14428
fired: [14428]
14428 fires away to 13984
{14428: 0, 13984: 74.0, 10286: 0.0, 11974: 0.0, 10680: 0.0, 164190: 0.0}
--- time step 2 ---
fired: [13984]
13984 fires away to 10286
{14428: 0.0, 13984: 0, 10286: 45.0, 11974: 0.0, 10680: 0.0, 164190: 0.0}
--- time step 3 ---
fired: [10286]
10286 fires away to 13984
10286 fires away to 11974
10286 fires away to 164190
{14428: 0.0, 13984: 9.0, 10286: 0, 11974: 92.0, 10680: 0.0, 164190: 10.0}
--- time step 4 ---
fired: [11974]
11974 fires away to 10286
11974 fires away to 10680
{14428: 0.0, 13984: 7.2, 10286: 4.0, 11974: 0, 10680: 41.0, 164190: 8.0}
--- time step 5 ---
fired: [10680]
10680 fires away to 11974
10680 fires away to 164190
{14428: 0.0, 13984: 5.76, 10286: 3.2, 11974: 5.0, 10680: 0, 164190: 84.4}
--- time step 6 ---
fired: [164190]
{14428: 0.0, 13984: 4.61, 10286: 2.56, 11974: 4.0, 10680: 0.0, 164190: 0}
--- time step 7 ---
fired: []
{14428: 0.0, 13984: 3.69, 10286: 2.05, 11974: 3.2, 10680

> Result: 164190 fires at tick 6, single pulse survives easily since 41 clears a threshold of 40.

Iteration 2. threshold = 42, single pulse, testing the bottleneck fails

In [ ]:
# reset for future iterations
charge = {nid: 0 for nid in neuron_ids}

decay_factor = 0.8
threshold = 42
time_steps = 8
external_input = {14428: 100}

for t in range(time_steps):
    print(f"--- time step {t+1} ---")

    # apply external input at tick 0
    if t == 0:
        for nid, amount in external_input.items():
            print("external applied to", nid)
            charge[nid] += amount

    # figure out who fires this tick, based on charge BEFORE any resets happen
    fired = [nid for nid in neuron_ids if charge[nid] >= threshold]
    print("fired:", fired)

    # apply the leak to everyone
    for nid in neuron_ids:
        charge[nid] *= decay_factor

    # firing neurons dump their charge to neighbors, then reset to 0
    for pre, post, weight in edges:
        if pre in fired:
            print(f"{pre} fires away to {post}")
            charge[post] += weight

    for nid in fired:
        charge[nid] = 0

    print({nid: round(c, 2) for nid, c in charge.items()})

--- time step 1 ---
external applied to 14428
fired: [14428]
14428 fires away to 13984
{14428: 0, 13984: 76.36, 10286: 1.31, 11974: 2.05, 10680: 0.0, 164190: 0.0}
--- time step 2 ---
fired: [13984]
13984 fires away to 10286
{14428: 0.0, 13984: 0, 10286: 46.05, 11974: 1.64, 10680: 0.0, 164190: 0.0}
--- time step 3 ---
fired: [10286]
10286 fires away to 13984
10286 fires away to 11974
10286 fires away to 164190
{14428: 0.0, 13984: 9.0, 10286: 0, 11974: 93.31, 10680: 0.0, 164190: 10.0}
--- time step 4 ---
fired: [11974]
11974 fires away to 10286
11974 fires away to 10680
{14428: 0.0, 13984: 7.2, 10286: 4.0, 11974: 0, 10680: 41.0, 164190: 8.0}
--- time step 5 ---
fired: []
{14428: 0.0, 13984: 5.76, 10286: 3.2, 11974: 0.0, 10680: 32.8, 164190: 6.4}
--- time step 6 ---
fired: []
{14428: 0.0, 13984: 4.61, 10286: 2.56, 11974: 0.0, 10680: 26.24, 164190: 5.12}
--- time step 7 ---
fired: []
{14428: 0.0, 13984: 3.69, 10286: 2.05, 11974: 0.0, 10680: 20.99, 164190: 4.1}
--- time step 8 ---
fired: []

> Result: 10680 stalls at 41, one short, signal never reaches 164190.

Iteration 3. two pulses, time_steps = 12

In [ ]:
# reset for future iterations
charge = {nid: 0 for nid in neuron_ids}

decay_factor = 0.8
threshold = 42
time_steps = 12
external_input = {14428: 100}

for t in range(time_steps):
    print(f"--- time step {t+1} ---")

    # apply external input at tick 0 and 4
    if t == 0 or t == 4:
        for nid, amount in external_input.items():
            print("external applied to", nid)
            charge[nid] += amount

    # figure out who fires this tick, based on charge BEFORE any resets happen
    fired = [nid for nid in neuron_ids if charge[nid] >= threshold]
    print("fired:", fired)

    # apply the leak to everyone
    for nid in neuron_ids:
        charge[nid] *= decay_factor

    # firing neurons dump their charge to neighbors, then reset to 0
    for pre, post, weight in edges:
        if pre in fired:
            print(f"{pre} fires away to {post}")
            charge[post] += weight

    for nid in fired:
        charge[nid] = 0

    print({nid: round(c, 2) for nid, c in charge.items()})

--- time step 1 ---
external applied to 14428
fired: [14428]
14428 fires away to 13984
{14428: 0, 13984: 76.36, 10286: 1.31, 11974: 0.0, 10680: 13.43, 164190: 2.62}
--- time step 2 ---
fired: [13984]
13984 fires away to 10286
{14428: 0.0, 13984: 0, 10286: 46.05, 11974: 0.0, 10680: 10.75, 164190: 2.1}
--- time step 3 ---
fired: [10286]
10286 fires away to 13984
10286 fires away to 11974
10286 fires away to 164190
{14428: 0.0, 13984: 9.0, 10286: 0, 11974: 92.0, 10680: 8.6, 164190: 11.68}
--- time step 4 ---
fired: [11974]
11974 fires away to 10286
11974 fires away to 10680
{14428: 0.0, 13984: 7.2, 10286: 4.0, 11974: 0, 10680: 47.88, 164190: 9.34}
--- time step 5 ---
external applied to 14428
fired: [14428, 10680]
10680 fires away to 11974
10680 fires away to 164190
14428 fires away to 13984
{14428: 0, 13984: 79.76, 10286: 3.2, 11974: 5.0, 10680: 0, 164190: 85.47}
--- time step 6 ---
fired: [13984, 164190]
13984 fires away to 10286
{14428: 0.0, 13984: 0, 10286: 47.56, 11974: 4.0, 10680: 0

> Result: 10680 fires at tick 5, 164190 fires at tick 6, confirming temporal summation.

Iteration 5. pushing the second pulse later, time_steps = 16, pulse at t=10

In [ ]:
# reset for future iterations
charge = {nid: 0 for nid in neuron_ids}

decay_factor = 0.8
threshold = 42
time_steps = 16
external_input = {14428: 100}

for t in range(time_steps):
    print(f"--- time step {t+1} ---")

    # apply external input at tick 0 and 10
    if t == 0 or t == 10:
        for nid, amount in external_input.items():
            print("external applied to", nid)
            charge[nid] += amount

    # figure out who fires this tick, based on charge BEFORE any resets happen
    fired = [nid for nid in neuron_ids if charge[nid] >= threshold]
    print("fired:", fired)

    # apply the leak to everyone
    for nid in neuron_ids:
        charge[nid] *= decay_factor

    # firing neurons dump their charge to neighbors, then reset to 0
    for pre, post, weight in edges:
        if pre in fired:
            print(f"{pre} fires away to {post}")
            charge[post] += weight

    for nid in fired:
        charge[nid] = 0

    print({nid: round(c, 2) for nid, c in charge.items()})

--- time step 1 ---
external applied to 14428
fired: [14428]
14428 fires away to 13984
{14428: 0, 13984: 76.36, 10286: 1.31, 11974: 0.0, 10680: 13.43, 164190: 2.62}
--- time step 2 ---
fired: [13984]
13984 fires away to 10286
{14428: 0.0, 13984: 0, 10286: 46.05, 11974: 0.0, 10680: 10.75, 164190: 2.1}
--- time step 3 ---
fired: [10286]
10286 fires away to 13984
10286 fires away to 11974
10286 fires away to 164190
{14428: 0.0, 13984: 9.0, 10286: 0, 11974: 92.0, 10680: 8.6, 164190: 11.68}
--- time step 4 ---
fired: [11974]
11974 fires away to 10286
11974 fires away to 10680
{14428: 0.0, 13984: 7.2, 10286: 4.0, 11974: 0, 10680: 47.88, 164190: 9.34}
--- time step 5 ---
fired: [10680]
10680 fires away to 11974
10680 fires away to 164190
{14428: 0.0, 13984: 5.76, 10286: 3.2, 11974: 5.0, 10680: 0, 164190: 85.47}
--- time step 6 ---
fired: [164190]
{14428: 0.0, 13984: 4.61, 10286: 2.56, 11974: 4.0, 10680: 0.0, 164190: 0}
--- time step 7 ---
fired: []
{14428: 0.0, 13984: 3.69, 10286: 2.05, 11974

> Result: still succeeds, 10680 fires tick 14, 164190 fires tick 15, this is what found the actual window boundary (works through gap 16, fails at gap 17).

Iteration 6. decay_factor = 0.9, same gap of 10, testing a slower leak

In [ ]:
# reset for future iterations
charge = {nid: 0 for nid in neuron_ids}

decay_factor = 0.9
threshold = 42
time_steps = 16
external_input = {14428: 100}

for t in range(time_steps):
    print(f"--- time step {t+1} ---")

    # apply external input at tick 0 and 10
    if t == 0 or t == 10:
        for nid, amount in external_input.items():
            print("external applied to", nid)
            charge[nid] += amount

    # figure out who fires this tick, based on charge BEFORE any resets happen
    fired = [nid for nid in neuron_ids if charge[nid] >= threshold]
    print("fired:", fired)

    # apply the leak to everyone
    for nid in neuron_ids:
        charge[nid] *= decay_factor

    # firing neurons dump their charge to neighbors, then reset to 0
    for pre, post, weight in edges:
        if pre in fired:
            print(f"{pre} fires away to {post}")
            charge[post] += weight

    for nid in fired:
        charge[nid] = 0

    print({nid: round(c, 2) for nid, c in charge.items()})

--- time step 1 ---
external applied to 14428
fired: [14428]
14428 fires away to 13984
{14428: 0, 13984: 78.15, 10286: 2.3, 11974: 0.0, 10680: 23.62, 164190: 4.61}
--- time step 2 ---
fired: [13984]
13984 fires away to 10286
{14428: 0.0, 13984: 0, 10286: 47.07, 11974: 0.0, 10680: 21.25, 164190: 4.15}
--- time step 3 ---
fired: [10286]
10286 fires away to 13984
10286 fires away to 11974
10286 fires away to 164190
{14428: 0.0, 13984: 9.0, 10286: 0, 11974: 92.0, 10680: 19.13, 164190: 13.73}
--- time step 4 ---
fired: [11974]
11974 fires away to 10286
11974 fires away to 10680
{14428: 0.0, 13984: 8.1, 10286: 4.0, 11974: 0, 10680: 58.22, 164190: 12.36}
--- time step 5 ---
fired: [10680]
10680 fires away to 11974
10680 fires away to 164190
{14428: 0.0, 13984: 7.29, 10286: 3.6, 11974: 5.0, 10680: 0, 164190: 89.12}
--- time step 6 ---
fired: [164190]
{14428: 0.0, 13984: 6.56, 10286: 3.24, 11974: 4.5, 10680: 0.0, 164190: 0}
--- time step 7 ---
fired: []
{14428: 0.0, 13984: 5.9, 10286: 2.92, 119

> Result: same tick timing (fires tick 15 either way), but far more leftover charge at the moment the second pulse lands (14.3 vs 4.4), confirming the window itself would be much wider (35 ticks vs 16) under this decay rate.

#### Analysis: firing vs a near miss

what's learned from this:
- firing clears a neuron's charge completely, a near miss doesn't, a neuron that almost reaches threshold keeps its leftover charge and just decays slowly
- that difference has a consequence for the rhythm game, a single isolated pulse and a rapid train of pulses behave very differently in this model, even though every individual pulse is identical

#### Temporal summation

new term: charge from events that are separated in time can still add up, as long as the leak hasn't drained it first. this only matters at a bottleneck neuron, one whose incoming drip is already close to threshold

#### The concept of the "timing window"

`10680` is the bottleneck: its rim (threshold) is 42, and each pulse that reaches it pours in 41, one short.

what happens with two pulses:
- the first pulse leaves `10680` at 41. it doesn't fire, but it isn't empty either, then the leak starts draining it
- when the second pulse arrives, it pours in another 41.

the question is, whether the water still left over, plus the 41, reaches 42 (so that it fires)

so the window is just: how long does the leftover stay at 1 or more?

past that gap, the leftover has drained too far and the second pulse is treated like a first one, and it fails the same way a single pulse would.

summary:
- the window is the longest gap between two pulses where the leftover, plus the new pour, still reaches the rim
- the window belongs to the weak link, not the whole path, `10680` is the only neuron here that needs leftover charge, because its pour (41) is below its rim (42)
- a higher rim shrinks the window, a slower leak widens it

how to calculate the window, i.e., how many ticks does the leftover survive before it drops below what's needed?
- the pour is 41 and the rim is 42, so the bucket needs at least 1 unit left over
- each tick multiplies the leftover by the decay factor
- so count how many ticks it takes for 41 to shrink below 1

In [32]:
leftover = 41
k = 0
while leftover * decay_factor >= 1:
    leftover *= decay_factor
    k += 1
print(k)

35


> with `decay_factor = 0.8` this prints 16, with `0.9` it prints 35. 35/16 * 100 = 218% increase in window size from a 0.1 change in decay
>
> formula version: `k = ln(41) / -ln(decay_factor)`, then round down
>
> the takeaway: the circuit remembers the first pulse for about 16 ticks (at `decay_factor = 0.8`) and can combine it with a later one, after that, the memory is gone.

#### What this means for input design

everything in the simulation turns into a design choice for stage 1:

- **the response has a fixed delay**: in this scaled LIF, a pulse takes 5 ticks to get from `14428` to `164190` (eyes to body), so to press something on beat, inject the input 5 ticks early
- **the threshold decides what one input can do**: at threshold <= 41, a single pulse always gets through, so the fly presses on any input. at threshold = 42, a single pulse fails, and two pulses are needed within the window. the second setup gives a natural design, an arrow appearing is the first pulse, the arrow reaching the hit line is the second, the fly only presses if the second follows the first closely enough
- **the window becomes the tempo tolerance**
- **a tick needs a real length**: it will be some number of ms, still undecided
- **where the external input is injected matters**: some photoreceptors reach 80,350 neurons and some reach only 5, so a well-connected one like `14428` has to be picked deliberately, not assumed

### Conclusion

what this file actually established:
- the real, found path survives: a pulse injected at `14428` does reach `164190`, confirming the structural path from `02_graph.ipynb` is also dynamically real, not just theoretically connected
- the path has a bottleneck (`11974 -> 10680`, weight 41), a single pulse can't cross it at `threshold = 42`, but two pulses close enough together can, through temporal summation
- that combining behavior only happens within a timing window, set by the bottleneck's margin and the decay rate, outside that window a second pulse is just treated as a fresh first pulse

this directly shapes the rhythm-game design: a fixed signal delay (5 ticks), a tunable tempo tolerance (the window), and a natural two-pulse design (arrow appears, arrow reaches the line) all fall out of mechanics that were discovered, not assumed going in

still open: the tick-to-millisecond mapping, which input encoding (visual vs auditory, from `04_further_exploration.ipynb`) actually gets used, and whether this behavior holds for other paths, not just this one